In [ ]:
# ==============================================================
# EXERCISE 9: HOPFIELD NEURAL NETWORK
# Associative Memory and Pattern Reconstruction System
# Dataset: Optical Recognition of Handwritten Digits (8x8)
# Run this entire program in ONE Google Colab cell.
# ==============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_digits
from collections import Counter

np.random.seed(42)

# ==============================================================
# 1. LOAD DATASET AND CREATE BIPOLAR DIGIT PATTERNS
# ==============================================================

digits_data = load_digits()

X_images = digits_data.images
y_digits = digits_data.target

# Convert 8x8 grayscale images into bipolar patterns.
# Pixel intensity > 8 becomes +1; otherwise -1.
def to_bipolar(image):
    return np.where(image.reshape(-1) > 8, 1, -1)

# Select one representative pattern for each digit.
stored_patterns = {}
representative_images = {}

for digit in range(10):
    index = np.where(y_digits == digit)[0][0]
    representative_images[digit] = X_images[index]
    stored_patterns[digit] = to_bipolar(X_images[index])

print("Optical Recognition of Handwritten Digits Dataset Loaded")
print("Total Images:", len(X_images))
print("Image Size: 8 x 8")
print("Stored Digit Patterns:", len(stored_patterns))


# ==============================================================
# 2. HOPFIELD NETWORK
# ==============================================================

class HopfieldNetwork:

    def __init__(self, patterns=None):
        self.patterns = {}
        self.weights = np.zeros((64, 64), dtype=float)

        if patterns:
            self.train(patterns)

    def train(self, patterns):
        self.patterns = {
            int(digit): np.asarray(pattern, dtype=int).copy()
            for digit, pattern in patterns.items()
        }

        self.weights = np.zeros((64, 64), dtype=float)

        # Hebbian learning rule.
        for pattern in self.patterns.values():
            self.weights += np.outer(pattern, pattern)

        # Remove self-connections.
        np.fill_diagonal(self.weights, 0)

        # Normalize weights.
        if len(self.patterns) > 0:
            self.weights /= len(self.patterns)

        return self

    def recall(self, input_pattern, max_iterations=100):
        state = np.asarray(input_pattern, dtype=int).copy()
        state = np.where(state >= 0, 1, -1)

        iterations = 0

        for iteration in range(max_iterations):
            previous_state = state.copy()

            # Asynchronous neuron updates.
            for neuron in np.random.permutation(len(state)):
                activation = np.dot(
                    self.weights[neuron], state
                )

                if activation > 0:
                    state[neuron] = 1
                elif activation < 0:
                    state[neuron] = -1

            iterations = iteration + 1

            if np.array_equal(state, previous_state):
                break

        return state, iterations

    def energy(self, pattern):
        return float(
            -0.5 * pattern @ self.weights @ pattern
        )

    def identify_digit(self, pattern):
        matches = []

        for digit, stored in self.patterns.items():
            if np.array_equal(pattern, stored):
                matches.append(digit)

        if matches:
            return matches[0]

        return None


# ==============================================================
# 3. HELPER FUNCTIONS
# ==============================================================

def get_network(digit_list=None):
    if digit_list is None:
        digit_list = list(stored_patterns.keys())

    patterns = {
        digit: stored_patterns[digit]
        for digit in digit_list
    }

    return HopfieldNetwork(patterns)


def display_pattern(pattern, title="Digit Pattern"):
    image = np.asarray(pattern).reshape(8, 8)

    plt.figure(figsize=(3, 3))
    plt.imshow(image, cmap="gray_r", vmin=-1, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()


def display_comparison(original, corrupted, recalled, digit):
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))

    images = [
        (original, "Original"),
        (corrupted, "Corrupted"),
        (recalled, "Recalled")
    ]

    for ax, (pattern, title) in zip(axes, images):
        ax.imshow(
            np.asarray(pattern).reshape(8, 8),
            cmap="gray_r",
            vmin=-1,
            vmax=1
        )
        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(f"Digit {digit}")
    plt.tight_layout()
    plt.show()


def corrupt_pattern(pattern, percentage, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    corrupted = pattern.copy()

    number_to_flip = int(round(len(pattern) * percentage / 100))
    number_to_flip = min(number_to_flip, len(pattern))

    indices = rng.choice(
        len(pattern),
        size=number_to_flip,
        replace=False
    )

    corrupted[indices] *= -1

    return corrupted, number_to_flip


def reconstruction_accuracy(original, recalled):
    return float(np.mean(original == recalled) * 100)


def make_network_for_count(count):
    digit_list = list(range(min(count, 10)))
    return get_network(digit_list)


def identify_closest_digit(pattern, digit_list=None):
    if digit_list is None:
        digit_list = list(stored_patterns.keys())

    distances = {
        digit: np.mean(pattern != stored_patterns[digit])
        for digit in digit_list
    }

    closest = min(distances, key=distances.get)

    # Require an exact pattern match to declare a recalled digit.
    if distances[closest] == 0:
        return closest

    return None


# ==============================================================
# MENU 1: STORE HANDWRITTEN DIGIT PATTERNS
# ==============================================================

def store_digit_patterns():
    print("\n--- STORE HANDWRITTEN DIGIT PATTERNS ---")

    network = get_network()

    print("Stored Digits:", list(network.patterns.keys()))
    print("Weight Matrix Shape:", network.weights.shape)
    print("\nWeight Matrix (first 10 rows and columns):")

    print(
        pd.DataFrame(
            network.weights[:10, :10]
        ).round(2).to_string(index=False)
    )

    for digit in range(10):
        display_pattern(
            stored_patterns[digit],
            f"Stored Digit {digit}"
        )

    print("All ten representative digit patterns are stored.")


# ==============================================================
# MENU 2: RECALL A NOISY DIGIT PATTERN
# ==============================================================

def recall_noisy_digit():
    print("\n--- RECALL A NOISY DIGIT PATTERN ---")

    try:
        digit = int(input("Enter digit (0-9): "))
        noise = float(
            input("Enter corruption percentage (5, 10, 20, 30): ")
        )

        if digit not in stored_patterns:
            print("Invalid digit.")
            return

        if noise not in [5, 10, 20, 30]:
            print("Choose 5, 10, 20, or 30.")
            return

        network = get_network()

        original = stored_patterns[digit].copy()

        corrupted, flipped = corrupt_pattern(
            original, noise
        )

        recalled, iterations = network.recall(corrupted)

        identified = network.identify_digit(recalled)

        print("\nOriginal Digit:", digit)
        print("Flipped Pixels:", flipped)
        print("Recall Iterations:", iterations)
        print(
            f"Pixel Reconstruction Accuracy: "
            f"{reconstruction_accuracy(original, recalled):.2f}%"
        )
        print("Identified Stored Digit:", identified)
        print("Final Energy:", network.energy(recalled))

        display_comparison(
            original, corrupted, recalled, digit
        )

    except ValueError:
        print("Enter valid numeric values.")


# ==============================================================
# MENU 3: MISSING PIXEL RECONSTRUCTION
# ==============================================================

def missing_pixel_challenge():
    print("\n--- MISSING PIXEL RECONSTRUCTION CHALLENGE ---")

    try:
        digit = int(input("Enter digit (0-9): "))

        if digit not in stored_patterns:
            print("Invalid digit.")
            return

        original = stored_patterns[digit].copy()
        corrupted = original.reshape(8, 8).copy()

        print("\n1. Remove a row")
        print("2. Remove a column")
        print("3. Remove a rectangular block")

        choice = input("Choose corruption type: ").strip()

        if choice == "1":
            row = int(input("Enter row number (0-7): "))
            if not 0 <= row <= 7:
                print("Invalid row.")
                return

            corrupted[row, :] = 0

        elif choice == "2":
            column = int(input("Enter column number (0-7): "))
            if not 0 <= column <= 7:
                print("Invalid column.")
                return

            corrupted[:, column] = 0

        elif choice == "3":
            row = int(input("Starting row (0-7): "))
            column = int(input("Starting column (0-7): "))
            height = int(input("Block height (1-8): "))
            width = int(input("Block width (1-8): "))

            if (
                not 0 <= row <= 7
                or not 0 <= column <= 7
                or not 1 <= height <= 8
                or not 1 <= width <= 8
                or row + height > 8
                or column + width > 8
            ):
                print("Invalid block dimensions.")
                return

            corrupted[
                row:row + height,
                column:column + width
            ] = 0

        else:
            print("Invalid choice.")
            return

        # Unknown pixels are initialized as inactive (-1).
        input_pattern = np.where(
            corrupted.reshape(-1) == 0,
            -1,
            corrupted.reshape(-1)
        )

        network = get_network()

        recalled, iterations = network.recall(input_pattern)

        accuracy = reconstruction_accuracy(
            original, recalled
        )

        print("\nDigit:", digit)
        print("Recall Iterations:", iterations)
        print(f"Reconstruction Accuracy: {accuracy:.2f}%")

        display_comparison(
            original,
            input_pattern,
            recalled,
            digit
        )

        if accuracy == 100:
            print("All pixels were reconstructed correctly.")
        else:
            print("Some pixels differ from the original pattern.")

        print(
            "Note: Missing pixels are initialized as -1; "
            "the Hopfield network does not explicitly distinguish "
            "unknown pixels from inactive pixels."
        )

    except ValueError:
        print("Enter valid numeric values.")


# ==============================================================
# MENU 4: DIGIT CORRUPTION TOLERANCE TEST
# ==============================================================

def corruption_tolerance_test():
    print("\n--- DIGIT CORRUPTION TOLERANCE TEST ---")

    network = get_network()
    rng = np.random.default_rng(42)

    levels = [5, 10, 20, 30, 40, 50]
    results = []

    for digit, original in stored_patterns.items():

        maximum_success = 0

        for level in levels:

            successful = 0
            trials = 10

            for _ in range(trials):

                corrupted, _ = corrupt_pattern(
                    original, level, rng
                )

                recalled, _ = network.recall(corrupted)

                if np.array_equal(recalled, original):
                    successful += 1

            success_rate = successful / trials * 100

            if success_rate > 0:
                maximum_success = level

        results.append({
            "Digit": digit,
            "Maximum Tested Successful Noise (%)": maximum_success
        })

    report = pd.DataFrame(results).sort_values(
        "Maximum Tested Successful Noise (%)",
        ascending=False
    )

    print(report.to_string(index=False))

    print(
        "\nRanking is based on the highest tested noise level "
        "with at least one successful recall among 10 trials."
    )


# ==============================================================
# MENU 5: HOPFIELD MEMORY CAPACITY CHALLENGE
# ==============================================================

def memory_capacity_challenge():
    print("\n--- HOPFIELD MEMORY CAPACITY CHALLENGE ---")

    rng = np.random.default_rng(42)

    counts = [2, 3, 4, 5, 6, 8, 10]
    results = []

    for count in counts:

        digit_list = list(range(count))
        network = get_network(digit_list)

        successful = 0
        total = 0

        for digit in digit_list:

            for _ in range(5):

                corrupted, _ = corrupt_pattern(
                    stored_patterns[digit], 10, rng
                )

                recalled, _ = network.recall(corrupted)

                if np.array_equal(
                    recalled, stored_patterns[digit]
                ):
                    successful += 1

                total += 1

        failed = total - successful
        accuracy = successful / total * 100

        results.append({
            "Stored Patterns": count,
            "Successful Recalls": successful,
            "Failed Recalls": failed,
            "Recall Accuracy (%)": accuracy
        })

        print(
            f"\nStored Patterns: {count}"
            f"\nSuccessful Recalls: {successful}"
            f"\nFailed Recalls: {failed}"
            f"\nRecall Accuracy: {accuracy:.2f}%"
        )

    report = pd.DataFrame(results)

    print("\n--- MEMORY CAPACITY REPORT ---")
    print(report.to_string(index=False))

    baseline = report.iloc[0]["Recall Accuracy (%)"]

    for _, row in report.iterrows():
        if row["Recall Accuracy (%)"] < baseline:
            print(
                "\nRecall performance first falls below the "
                f"{int(report.iloc[0]['Stored Patterns'])}-pattern "
                f"baseline at {int(row['Stored Patterns'])} patterns."
            )
            break
    else:
        print(
            "\nNo configuration fell below the first configuration's "
            "recall accuracy."
        )


# ==============================================================
# MENU 6: FALSE MEMORY DETECTOR
# ==============================================================

def false_memory_detector():
    print("\n--- FALSE MEMORY DETECTOR ---")

    try:
        digit = int(input("Enter digit for noisy test (0-9): "))

        if digit not in stored_patterns:
            print("Invalid digit.")
            return

        print("\n1. Corrupted stored digit")
        print("2. Random bipolar pattern")

        choice = input("Choose input type: ").strip()

        if choice == "1":

            noise = float(
                input("Enter corruption percentage: ")
            )

            if not 0 <= noise <= 100:
                print("Noise must be between 0 and 100.")
                return

            input_pattern, flipped = corrupt_pattern(
                stored_patterns[digit], noise
            )

        elif choice == "2":

            input_pattern = np.random.choice(
                [-1, 1], size=64
            )
            flipped = 0

        else:
            print("Invalid choice.")
            return

        network = get_network()

        recalled, iterations = network.recall(input_pattern)

        matched_digit = network.identify_digit(recalled)

        false_memory = matched_digit is None

        print("\nInput Pattern:")
        print(input_pattern.reshape(8, 8))

        print("\nFinal Stable Pattern:")
        print(recalled.reshape(8, 8))

        print("\nRecall Iterations:", iterations)
        print("Input Pixels Flipped:", flipped)
        print("Matching Stored Digit:", matched_digit)

        if false_memory:
            print("Result: SPURIOUS / FALSE MEMORY")
        else:
            print(f"Result: Matches Stored Digit {matched_digit}")

        display_pattern(input_pattern, "Input Pattern")
        display_pattern(recalled, "Final Recalled Pattern")

    except ValueError:
        print("Enter valid numeric values.")


# ==============================================================
# MENU 7: DIGIT CONFUSION ANALYZER
# ==============================================================

def digit_confusion_analyzer():
    print("\n--- DIGIT CONFUSION ANALYZER ---")

    network = get_network()
    rng = np.random.default_rng(42)

    confusion = np.zeros((10, 11), dtype=int)

    # Columns 0-9 represent exact stored-digit matches.
    # Column 10 represents false/spurious memories.
    trials = 20
    noise_level = 20

    for digit in range(10):

        original = stored_patterns[digit]

        for _ in range(trials):

            corrupted, _ = corrupt_pattern(
                original, noise_level, rng
            )

            recalled, _ = network.recall(corrupted)

            matched_digit = network.identify_digit(recalled)

            if matched_digit is None:
                confusion[digit, 10] += 1
            else:
                confusion[digit, matched_digit] += 1

    columns = [str(digit) for digit in range(10)] + ["False Memory"]

    report = pd.DataFrame(
        confusion,
        index=[f"Actual {digit}" for digit in range(10)],
        columns=columns
    )

    print(f"\nConfusion Matrix ({noise_level}% corruption, {trials} trials per digit)")
    print(report)

    pairs = []

    for actual in range(10):
        for predicted in range(10):
            if actual != predicted:
                count = confusion[actual, predicted]

                if count > 0:
                    pairs.append(
                        (actual, predicted, int(count))
                    )

    pairs.sort(key=lambda item: item[2], reverse=True)

    print("\nMost Frequently Confused Digit Pairs:")

    if pairs:
        for actual, predicted, count in pairs[:10]:
            print(
                f"Digit {actual} -> Digit {predicted}: "
                f"{count} times"
            )
    else:
        print("No incorrect stored-digit matches detected.")

    print(
        "\nNote: A recalled pattern that does not exactly match "
        "any stored digit is counted as a false memory."
    )


# ==============================================================
# MENU 8: HOPFIELD RECALL PERFORMANCE REPORT
# ==============================================================

def recall_performance_report():
    print("\n--- HOPFIELD RECALL PERFORMANCE REPORT ---")

    network = get_network()
    rng = np.random.default_rng(42)

    results = []

    for digit, original in stored_patterns.items():

        for level in [5, 10, 20, 30, 40, 50]:

            corrupted, flipped = corrupt_pattern(
                original, level, rng
            )

            recalled, iterations = network.recall(corrupted)

            accuracy = reconstruction_accuracy(
                original, recalled
            )

            matched_digit = network.identify_digit(recalled)

            results.append({
                "Stored Digit": digit,
                "Corruption (%)": level,
                "Flipped Pixels": flipped,
                "Recall Iterations": iterations,
                "Pixel Accuracy (%)": accuracy,
                "Recalled Digit": (
                    matched_digit
                    if matched_digit is not None
                    else "False Memory"
                ),
                "False Memory": matched_digit is None
            })

    report = pd.DataFrame(results)

    print(report.to_string(index=False))

    print("\nOverall Results:")
    print("Total Experiments:", len(report))
    print(
        "Exact Stored-Pattern Recalls:",
        int((~report["False Memory"]).sum())
    )
    print(
        "False Memories:",
        int(report["False Memory"].sum())
    )
    print(
        f"Average Pixel Reconstruction Accuracy: "
        f"{report['Pixel Accuracy (%)'].mean():.2f}%"
    )

    print("\nAverage Accuracy by Corruption Level:")

    summary = report.groupby("Corruption (%)").agg(
        Average_Accuracy=("Pixel Accuracy (%)", "mean"),
        False_Memories=("False Memory", "sum")
    )

    print(summary.round(2))


# ==============================================================
# MAIN MENU
# ==============================================================

def main():

    while True:

        print("\n" + "=" * 65)
        print(" HOPFIELD NEURAL NETWORK ASSOCIATIVE MEMORY SYSTEM")
        print("=" * 65)

        print("1. Store Handwritten Digit Patterns")
        print("2. Recall a Noisy Digit Pattern")
        print("3. Missing Pixel Reconstruction Challenge")
        print("4. Digit Corruption Tolerance Test")
        print("5. Hopfield Memory Capacity Challenge")
        print("6. False Memory Detector")
        print("7. Digit Confusion Analyzer")
        print("8. Hopfield Recall Performance Report")
        print("9. Exit")

        choice = input("\nEnter your choice (1-9): ").strip()

        try:

            if choice == "1":
                store_digit_patterns()

            elif choice == "2":
                recall_noisy_digit()

            elif choice == "3":
                missing_pixel_challenge()

            elif choice == "4":
                corruption_tolerance_test()

            elif choice == "5":
                memory_capacity_challenge()

            elif choice == "6":
                false_memory_detector()

            elif choice == "7":
                digit_confusion_analyzer()

            elif choice == "8":
                recall_performance_report()

            elif choice == "9":
                print("Exiting Hopfield Associative Memory System.")
                break

            else:
                print("Invalid choice. Please enter 1-9.")

        except Exception as error:
            print("Error:", error)


main()

Optical Recognition of Handwritten Digits Dataset Loaded
Total Images: 1797
Image Size: 8 x 8
Stored Digit Patterns: 10

 HOPFIELD NEURAL NETWORK ASSOCIATIVE MEMORY SYSTEM
1. Store Handwritten Digit Patterns
2. Recall a Noisy Digit Pattern
3. Missing Pixel Reconstruction Challenge
4. Digit Corruption Tolerance Test
5. Hopfield Memory Capacity Challenge
6. False Memory Detector
7. Digit Confusion Analyzer
8. Hopfield Recall Performance Report
9. Exit

Enter your choice (1-9): 9
Exiting Hopfield Associative Memory System.
